<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/Structured_Output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Structured Output using OpenAI APIs

In [2]:
!pip install -q openai

In [3]:
import os
from openai import OpenAI

from google.colab import userdata
from pydantic import BaseModel

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
OPENAI_AI_MODEL = 'gpt-5.6-luna'

# OpenAI Task 1

In [4]:
CLASSIFIER_PROMPT = """
You are a helpful assistant that classifies user messages into categories.
Classify the message into one of the following categories:
- smalltalk
- complaint
- status
- feedback

If you don't know the category, classify it as "smalltalk".
Also, for complaints, assign severity level
- 1 if this requires immediate response
- 2 if this requires response within two business dats
- 3 if this can be ignored
"""

In [5]:
class CategorySeverity(BaseModel):
    category: str
    severity: int

In [6]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

user_request = "My Chiller is getting noisy and needs to be replaced soon."

response = client.responses.parse(
    model=OPENAI_AI_MODEL,
    input=CLASSIFIER_PROMPT+user_request,
    text_format=CategorySeverity,
)

chat_information = response.output_parsed  # A NewsExtraction instance



In [10]:
print("OpenAI Structured Output Result:")
print(f"Category:   {chat_information.category}")
print(f"Severity:   {chat_information.severity}")
if chat_information.category == "complaint":
  if chat_information.severity == 1:
    print(f"Send Complaint to emergency_support@company.com:  {user_request} (Requires immediate answer)")
  elif chat_information.severity == 2:
    print(f"Send Complaint to regular_support@company.com :  {user_request} (Needs response within two business days)")
  elif chat_information.severity == 3:
    print(f"Complaint:  {user_request} (Ignore)")
elif chat_information.category == "feedback":
    print(f"Feedback:   {user_request}")
elif chat_information.category == "status":
    print(f"Status:     {user_request}")
else:
    print(f"Chat:       {user_request}")

OpenAI Structured Output Result:
Category:   complaint
Severity:   2
Send Complaint to regular_support@company.com :  My Chiller is getting noisy and needs to be replaced soon. (Needs response within two business days)


# OpenAI Task 2

In [ ]:
system_prompt = """
You are an expert at structured data extraction. You will be given unstructured
text from a news article and should convert it into the given structure.
"""

news_article = """
With an estimated $750 million jackpot, Monday’s Powerball draw will be the 10th
largest in the game’s history after no player matched all six number in
Saturday’s draw, the Multi-State Lottery Association said in a press release.
"""

In [ ]:
class NewsExtraction(BaseModel):
    lottery: str
    jackpot: int
    organizer: str

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.responses.parse(
    model=OPENAI_AI_MODEL,
    input=system_prompt+news_article,
    text_format=NewsExtraction,
)

news_information = response.output_parsed  # A NewsExtraction instance

print("OpenAI Structured Output Result:")
print(f"Lottery:   {news_information.lottery}")
print(f"Jackpot:   {news_information.jackpot}")
print(f"Organizer: {news_information.organizer}")
# print(f"Data Type: {type(news_information)}")


OpenAI Structured Output Result:
Lottery:   Powerball
Jackpot:   750000000
Organizer: Multi-State Lottery Association


# Structured Output with Google Gemini

In [11]:
!pip install -q google-genai

In [12]:
import os
from google import genai
from google.genai import types

from google.colab import userdata
from pydantic import BaseModel

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
GEMINI_AI_MODEL = "gemini-2.5-flash"

# Google Gemini Task 1

In [13]:
CLASSIFIER_PROMPT = """
You are a helpful assistant that classifies user messages into categories.
Classify the message into one of the following categories:
- smalltalk
- complaint
- status
- feedback

If you don't know the category, classify it as "smalltalk".
Also, for complaints, assign severity level
- 1 if this requires immediate response
- 2 if this requires response within two business dats
- 3 if this can be ignored
"""

In [14]:
class CategorySeverity(BaseModel):
    category: str
    severity: int

In [16]:
chat_information = response.output_parsed  # A NewsExtraction instance

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

user_request = "My Chiller is getting noisy and needs to be replaced soon."

response = client.models.generate_content(
    model=GEMINI_AI_MODEL,
    contents=CLASSIFIER_PROMPT+user_request,
    config=types.GenerateContentConfig(
        temperature=0,
        response_mime_type="application/json",
        response_schema=CategorySeverity,
    ),
)

chat_information = response.parsed

In [17]:
print("Gemini Structured Output Result:")
print(f"Category:   {chat_information.category}")
print(f"Severity:   {chat_information.severity}")
if chat_information.category == "complaint":
  if chat_information.severity == 1:
    print(f"Send Complaint to emergency_support@company.com:  {user_request} (Requires immediate answer)")
  elif chat_information.severity == 2:
    print(f"Send Complaint to regular_support@company.com :  {user_request} (Needs response within two business days)")
  elif chat_information.severity == 3:
    print(f"Complaint:  {user_request} (Ignore)")
elif chat_information.category == "feedback":
    print(f"Feedback:   {user_request}")
elif chat_information.category == "status":
    print(f"Status:     {user_request}")
else:
    print(f"Chat:       {user_request}")

Gemini Structured Output Result:
Category:   complaint
Severity:   2
Send Complaint to regular_support@company.com :  My Chiller is getting noisy and needs to be replaced soon. (Needs response within two business days)


# Google Gemini Task 2

In [ ]:
class NewsExtraction(BaseModel):
    lottery: str | None = None
    jackpot: int | None = None
    organizer: str | None = None

In [ ]:
system_prompt = """
You are an expert at structured data extraction. You will be given unstructured
text from a news article and should convert it into the given structure.
"""

news_article = """
With an estimated $750 million jackpot, Monday’s Powerball draw will be the 10th
largest in the game’s history after no player matched all six number in
Saturday’s draw, the Multi-State Lottery Association said in a press release.
"""

In [ ]:
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model=GEMINI_AI_MODEL,
    contents=system_prompt+news_article,
    config=types.GenerateContentConfig(
        temperature=0,
        response_mime_type="application/json",
        response_schema=NewsExtraction,
    ),
)

news_information = response.parsed

print("Gemini Structured Output Result:")
print(f"Lottery:   {news_information.lottery}")
print(f"Jackpot:   {news_information.jackpot}")
print(f"Organizer: {news_information.organizer}")
#print(f"Data Type: {type(news_information)}")

Gemini Structured Output Result:
Lottery:   Powerball
Jackpot:   750000000
Organizer: Multi-State Lottery Association
